# Supplementary Analysis — Novelty 3 Gap Fill
## Engagement Metrics Comparison per Narrative (BERTopic-Validated)

**Tujuan:**  
Membuktikan secara eksplisit bahwa narasi Affan Kurniawan menghasilkan *diffusion intensity* yang
signifikan lebih tinggi dibanding narasi ekonomi (Demo & DPR), diukur dari metrik mentah:
`retweet_count`, `reply_count`, `quote_count`, `view_count`, `favorite_count`.

**Output yang dihasilkan:**
1. Tabel komparatif median & mean per narasi per metrik  
2. Kruskal-Wallis test (overall group difference)  
3. Post-hoc Mann-Whitney pairwise (Affan vs Demo&DPR, Affan vs Kekerasan, dll.)  
4. Effect size (rank-biserial correlation r)  
5. File CSV output siap masuk Results section paper

**Input file:** `Data_with_community3.csv`  
**Kolom kunci:** `narrative`, `retweet_count`, `reply_count`, `quote_count`, `view_count`, `favorite_count`

In [19]:
# Cell 1 — Install & Import
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [20]:
# Cell 2 — Load Data
df = pd.read_csv('Data_with_community3.csv', sep=';')

print(f'Total rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'\nNarrative distribution:')
print(df['narrative'].value_counts())

Total rows: 14,248
Columns: ['date', 'is_quote', 'mentions', 'username', 'verified', 'account_type', 'full_text', 'tweet_url', 'quoted_url', 'view_count', 'quote_count', 'quoted_text', 'reply_count', 'display_name', 'retweet_count', 'favorite_count', 'in_reply_to_url', 'quoted_username', 'user_statuses_count', 'user_followers_count', 'user_following_count', 'in_reply_to_screen_name', 'topic', 'probability', 'topic_label', 'narrative', 'username_clean', 'community']

Narrative distribution:
narrative
Demo & DPR            2482
Gerakan/Hashtag       2356
Kekerasan Aparat      2276
Politik & Tuntutan    1437
Ekonomi Rakyat        1277
Affan Kurniawan        743
Keamanan & Respons     206
Name: count, dtype: int64


In [21]:
# Cell 3 — Define engagement columns & validate
ENGAGEMENT_COLS = ['retweet_count', 'reply_count', 'quote_count', 'view_count', 'favorite_count']

# Cek ketersediaan kolom
missing = [c for c in ENGAGEMENT_COLS if c not in df.columns]
available = [c for c in ENGAGEMENT_COLS if c in df.columns]

print(f'Available engagement columns: {available}')
if missing:
    print(f'Missing (will be skipped): {missing}')

# Gunakan hanya yang tersedia
ENGAGEMENT_COLS = available

# Pastikan numerik
for col in ENGAGEMENT_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows di mana semua engagement = NaN
df_eng = df.dropna(subset=ENGAGEMENT_COLS, how='all').copy()
print(f'\nRows with at least one engagement value: {len(df_eng):,} (dari {len(df):,})')

# Tampilkan basic stats per engagement column
print('\nGlobal stats per metric:')
print(df_eng[ENGAGEMENT_COLS].describe().round(2))

Available engagement columns: ['retweet_count', 'reply_count', 'quote_count', 'view_count', 'favorite_count']

Rows with at least one engagement value: 11,090 (dari 14,248)

Global stats per metric:
       retweet_count  reply_count  quote_count   view_count  favorite_count
count       11090.00     11090.00     11090.00     11090.00        11090.00
mean          213.51        12.98         8.43     28070.88          464.93
std          4599.88       100.66       166.63    341371.68         6653.35
min             0.00         0.00         0.00         0.00            0.00
25%             0.00         0.00         0.00       440.00            0.00
50%             0.00         0.00         0.00      1180.00            0.00
75%             0.00         0.00         0.00      4460.00           20.00
max        355200.00      4150.00      9340.00  21353610.00       363380.00


In [22]:
# Cell 4 — Descriptive Statistics per Narrative
# Hitung mean, median, std, count per narasi per metrik

narratives = df_eng['narrative'].dropna().unique()
print(f'Narratives: {narratives}\n')

rows = []
for narr in sorted(narratives):
    sub = df_eng[df_eng['narrative'] == narr]
    row = {'narrative': narr, 'n_tweets': len(sub)}
    for col in ENGAGEMENT_COLS:
        row[f'{col}_median'] = sub[col].median()
        row[f'{col}_mean']   = sub[col].mean()
        row[f'{col}_std']    = sub[col].std()
    rows.append(row)

desc_df = pd.DataFrame(rows)

# Tampilkan dalam format pivot yang mudah dibaca
print('=== MEDIAN per Narrative ===')
median_cols = ['narrative', 'n_tweets'] + [f'{c}_median' for c in ENGAGEMENT_COLS]
print(desc_df[median_cols].to_string(index=False))

print('\n=== MEAN per Narrative ===')
mean_cols = ['narrative', 'n_tweets'] + [f'{c}_mean' for c in ENGAGEMENT_COLS]
print(desc_df[mean_cols].round(2).to_string(index=False))

Narratives: ['Demo & DPR' 'Ekonomi Rakyat' 'Kekerasan Aparat' 'Gerakan/Hashtag'
 'Politik & Tuntutan' 'Affan Kurniawan' 'Keamanan & Respons']

=== MEDIAN per Narrative ===
         narrative  n_tweets  retweet_count_median  reply_count_median  quote_count_median  view_count_median  favorite_count_median
   Affan Kurniawan       714                   0.0                 0.0                 0.0             3750.0                   10.0
        Demo & DPR      1614                   0.0                 0.0                 0.0             1360.0                    0.0
    Ekonomi Rakyat      1070                   0.0                 0.0                 0.0              820.0                    0.0
   Gerakan/Hashtag      2043                   0.0                 0.0                 0.0              980.0                    0.0
Keamanan & Respons       179                   0.0                 0.0                 0.0             2640.0                   10.0
  Kekerasan Aparat      1932  

In [23]:
# Cell 5 — Kruskal-Wallis Test (Overall Group Differences)
# Non-parametric ANOVA — sesuai distribusi engagement yang skewed

print('=== KRUSKAL-WALLIS TEST per Engagement Metric ===\n')
print(f'{"Metric":<20} {"H-statistic":>12} {"p-value":>12} {"Significant?":>14}')
print('-' * 62)

kruskal_results = {}
for col in ENGAGEMENT_COLS:
    groups = []
    for narr in sorted(narratives):
        g = df_eng[df_eng['narrative'] == narr][col].dropna().values
        if len(g) > 0:
            groups.append(g)

    if len(groups) >= 2:
        h_stat, p_val = stats.kruskal(*groups)
        sig = '*** p<0.001' if p_val < 0.001 else ('** p<0.01' if p_val < 0.01 else ('* p<0.05' if p_val < 0.05 else 'n.s.'))
        kruskal_results[col] = {'H': h_stat, 'p': p_val, 'sig': sig}
        print(f'{col:<20} {h_stat:>12.3f} {p_val:>12.6f} {sig:>14}')

print('\nNote: Kruskal-Wallis tidak mengasumsikan normalitas — sesuai untuk engagement data yang heavy-tailed.')

=== KRUSKAL-WALLIS TEST per Engagement Metric ===

Metric                H-statistic      p-value   Significant?
--------------------------------------------------------------
retweet_count             243.561     0.000000    *** p<0.001
reply_count               203.712     0.000000    *** p<0.001
quote_count               157.394     0.000000    *** p<0.001
view_count                370.287     0.000000    *** p<0.001
favorite_count            221.968     0.000000    *** p<0.001

Note: Kruskal-Wallis tidak mengasumsikan normalitas — sesuai untuk engagement data yang heavy-tailed.


In [24]:
# Cell 6 — Post-hoc Pairwise Mann-Whitney U (with Effect Size + Bonferroni)
# Bandingkan setiap pasang narasi per metrik
# Effect size: rank-biserial correlation r = 1 - (2*U)/(n1*n2)
# Bonferroni: koreksi global lintas SEMUA test (n_metrics x n_pairs)

print('=== PAIRWISE MANN-WHITNEY U TEST (dengan Effect Size + Bonferroni) ===\n')

narrative_list = sorted(narratives)
pairs = list(combinations(narrative_list, 2))

# --- Hitung total jumlah test untuk Bonferroni global ---
# Hanya hitung pair yang valid (n>=5 di kedua grup) per metrik
valid_test_count = 0
for col in ENGAGEMENT_COLS:
    for (n1, n2) in pairs:
        g1 = df_eng[df_eng['narrative'] == n1][col].dropna().values
        g2 = df_eng[df_eng['narrative'] == n2][col].dropna().values
        if len(g1) >= 5 and len(g2) >= 5:
            valid_test_count += 1

ALPHA = 0.05
M_TESTS = valid_test_count
ALPHA_BONF = ALPHA / M_TESTS

print(f'Total valid tests: {M_TESTS}  ({len(ENGAGEMENT_COLS)} metrics x {len(pairs)} pairs, dikurangi yang n<5)')
print(f'Bonferroni-corrected alpha: {ALPHA}/{M_TESTS} = {ALPHA_BONF:.6f}\n')

pairwise_rows = []

for col in ENGAGEMENT_COLS:
    print(f'--- Metric: {col} ---')
    print(f'{"Pair":<55} {"U":>10} {"p_raw":>10} {"p_bonf":>10} {"r":>8} {"Sig_raw":>9} {"Sig_bonf":>10}')
    print('-' * 116)

    for (n1, n2) in pairs:
        g1 = df_eng[df_eng['narrative'] == n1][col].dropna().values
        g2 = df_eng[df_eng['narrative'] == n2][col].dropna().values

        if len(g1) < 5 or len(g2) < 5:
            continue

        u_stat, p_val = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        r = 1 - (2 * u_stat) / (len(g1) * len(g2))

        # Bonferroni-adjusted p-value (capped at 1.0)
        p_bonf = min(p_val * M_TESTS, 1.0)

        # Raw significance
        if p_val < 0.001:
            sig_raw = '***'
        elif p_val < 0.01:
            sig_raw = '**'
        elif p_val < 0.05:
            sig_raw = '*'
        else:
            sig_raw = 'n.s.'

        # Bonferroni-adjusted significance (compare raw p to alpha_bonf)
        if p_val < ALPHA_BONF:
            sig_bonf = 'sig'
        else:
            sig_bonf = 'n.s.'

        pair_label = f'{n1} vs {n2}'
        print(f'{pair_label:<55} {u_stat:>10.0f} {p_val:>10.4g} {p_bonf:>10.4g} {r:>8.3f} {sig_raw:>9} {sig_bonf:>10}')

        pairwise_rows.append({
            'metric': col,
            'narrative_1': n1,
            'narrative_2': n2,
            'n1': len(g1),
            'n2': len(g2),
            'U_statistic': round(u_stat, 2),
            'p_value_raw': float(f'{p_val:.6g}'),
            'p_value_bonferroni': float(f'{p_bonf:.6g}'),
            'alpha_bonferroni': round(ALPHA_BONF, 6),
            'effect_size_r': round(r, 3),
            'significant_raw': sig_raw,
            'significant_bonferroni': sig_bonf
        })
    print()

print('Effect size: r < 0.1 negligible, 0.1-0.3 small, 0.3-0.5 medium, > 0.5 large')
print(f'Bonferroni rule: reject H0 jika p_raw < {ALPHA_BONF:.6f} (ekivalen p_bonf < 0.05)')


=== PAIRWISE MANN-WHITNEY U TEST (dengan Effect Size + Bonferroni) ===

Total valid tests: 105  (5 metrics x 21 pairs, dikurangi yang n<5)
Bonferroni-corrected alpha: 0.05/105 = 0.000476

--- Metric: retweet_count ---
Pair                                                             U      p_raw     p_bonf        r   Sig_raw   Sig_bonf
--------------------------------------------------------------------------------------------------------------------
Affan Kurniawan vs Demo & DPR                               707654  8.016e-35  8.417e-33   -0.228       ***        sig
Affan Kurniawan vs Ekonomi Rakyat                           464934  1.817e-25  1.908e-23   -0.217       ***        sig
Affan Kurniawan vs Gerakan/Hashtag                          858208   6.69e-21  7.025e-19   -0.177       ***        sig
Affan Kurniawan vs Keamanan & Respons                        68863    0.05906          1   -0.078      n.s.       n.s.
Affan Kurniawan vs Kekerasan Aparat                         851064  5.

In [25]:
# Cell 6b — Bonferroni Summary: Which tests survive correction?
# Tabel ringkas yang siap dimasukkan ke Results section

pairwise_df_preview = pd.DataFrame(pairwise_rows)

print('=== BONFERRONI SURVIVAL SUMMARY ===\n')
print(f'Total tests dijalankan       : {len(pairwise_df_preview)}')
print(f'Bonferroni-corrected alpha   : {ALPHA_BONF:.6f}')
print(f'Tests significant (raw p<.05): {(pairwise_df_preview["significant_raw"] != "n.s.").sum()}')
print(f'Tests survive Bonferroni     : {(pairwise_df_preview["significant_bonferroni"] == "sig").sum()}')
print(f'Tests lost to correction     : {((pairwise_df_preview["significant_raw"] != "n.s.") & (pairwise_df_preview["significant_bonferroni"] == "n.s.")).sum()}')

print('\n=== TESTS YANG SURVIVE BONFERRONI (p_raw < {:.6f}) ==='.format(ALPHA_BONF))
survived = pairwise_df_preview[pairwise_df_preview['significant_bonferroni'] == 'sig'].copy()
survived = survived.sort_values(['metric', 'p_value_raw'])

if len(survived) > 0:
    display_cols = ['metric', 'narrative_1', 'narrative_2', 'n1', 'n2',
                    'p_value_raw', 'p_value_bonferroni', 'effect_size_r']
    print(survived[display_cols].to_string(index=False))
else:
    print('Tidak ada test yang survive koreksi Bonferroni.')

print('\n=== TESTS YANG GUGUR SETELAH BONFERRONI (sig raw, tapi n.s. setelah koreksi) ===')
lost = pairwise_df_preview[
    (pairwise_df_preview['significant_raw'] != 'n.s.') &
    (pairwise_df_preview['significant_bonferroni'] == 'n.s.')
].copy()
lost = lost.sort_values(['metric', 'p_value_raw'])

if len(lost) > 0:
    print(lost[['metric', 'narrative_1', 'narrative_2', 'p_value_raw', 'p_value_bonferroni', 'effect_size_r']].to_string(index=False))
else:
    print('(tidak ada)')


=== BONFERRONI SURVIVAL SUMMARY ===

Total tests dijalankan       : 105
Bonferroni-corrected alpha   : 0.000476
Tests significant (raw p<.05): 77
Tests survive Bonferroni     : 57
Tests lost to correction     : 20

=== TESTS YANG SURVIVE BONFERRONI (p_raw < 0.000476) ===
        metric        narrative_1        narrative_2   n1   n2  p_value_raw  p_value_bonferroni  effect_size_r
favorite_count    Affan Kurniawan   Kekerasan Aparat  714 1932 1.454280e-35        1.527000e-33         -0.275
favorite_count    Affan Kurniawan         Demo & DPR  714 1614 3.100150e-25        3.255160e-23         -0.241
favorite_count    Affan Kurniawan     Ekonomi Rakyat  714 1070 1.226940e-23        1.288280e-21         -0.252
favorite_count    Gerakan/Hashtag   Kekerasan Aparat 2043 1932 1.007460e-15        1.057830e-13         -0.129
favorite_count    Affan Kurniawan    Gerakan/Hashtag  714 2043 4.275210e-14        4.488970e-12         -0.175
favorite_count   Kekerasan Aparat Politik & Tuntutan 1932  958

In [26]:
# Cell 7 — Ringkasan Fokus: Affan Kurniawan vs Narasi Lain (dengan Bonferroni)
# Ini yang langsung mendukung Novelty 3 — tabel siap masuk paper

print('=== FOKUS: Affan Kurniawan vs Narasi Lain ===\n')

# Identifikasi label narasi Affan
affan_candidates = [n for n in narrative_list if 'affan' in n.lower() or 'kurniawan' in n.lower()]

print(f'Narrative candidates untuk Affan: {affan_candidates}')
print(f'Semua narratives: {narrative_list}')

if affan_candidates:
    AFFAN_NARRATIVE = affan_candidates[0]
    print(f'\nMenggunakan: "{AFFAN_NARRATIVE}"')
    print(f'Bonferroni alpha (global, {M_TESTS} tests): {ALPHA_BONF:.6f}\n')

    affan_data = df_eng[df_eng['narrative'] == AFFAN_NARRATIVE]
    others = [n for n in narrative_list if n != AFFAN_NARRATIVE]

    print(f'Affan Kurniawan (n={len(affan_data):,}) median engagement:')
    for col in ENGAGEMENT_COLS:
        print(f'  {col}: {affan_data[col].median():.1f}')

    print()
    for other_narr in others:
        other_data = df_eng[df_eng['narrative'] == other_narr]
        print(f'vs {other_narr} (n={len(other_data):,}):')
        for col in ENGAGEMENT_COLS:
            g1 = affan_data[col].dropna().values
            g2 = other_data[col].dropna().values
            if len(g1) >= 5 and len(g2) >= 5:
                u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
                r = 1 - (2 * u) / (len(g1) * len(g2))
                p_bonf = min(p * M_TESTS, 1.0)
                med_a = affan_data[col].median()
                med_o = other_data[col].median()
                ratio = med_a / (med_o + 1e-9)
                # Bonferroni-based marker
                if p < ALPHA_BONF:
                    sig_mark = 'sig (Bonf)'
                elif p < 0.05:
                    sig_mark = 'sig (raw only)'
                else:
                    sig_mark = 'n.s.'
                print(f'  {col}: Affan median={med_a:.1f} vs {med_o:.1f} | ratio={ratio:.1f}x | r={r:.3f} | p_raw={p:.4g} | p_bonf={p_bonf:.4g} | {sig_mark}')
        print()
else:
    print('\n[!] Tidak ditemukan narasi Affan secara otomatis.')
    print('Set manual: AFFAN_NARRATIVE = "<nama narasi di data>" lalu jalankan ulang cell ini.')


=== FOKUS: Affan Kurniawan vs Narasi Lain ===

Narrative candidates untuk Affan: ['Affan Kurniawan']
Semua narratives: ['Affan Kurniawan', 'Demo & DPR', 'Ekonomi Rakyat', 'Gerakan/Hashtag', 'Keamanan & Respons', 'Kekerasan Aparat', 'Politik & Tuntutan']

Menggunakan: "Affan Kurniawan"
Bonferroni alpha (global, 105 tests): 0.000476

Affan Kurniawan (n=714) median engagement:
  retweet_count: 0.0
  reply_count: 0.0
  quote_count: 0.0
  view_count: 3750.0
  favorite_count: 10.0

vs Demo & DPR (n=1,614):
  retweet_count: Affan median=0.0 vs 0.0 | ratio=0.0x | r=-0.228 | p_raw=8.016e-35 | p_bonf=8.417e-33 | sig (Bonf)
  reply_count: Affan median=0.0 vs 0.0 | ratio=0.0x | r=-0.117 | p_raw=3.511e-08 | p_bonf=3.687e-06 | sig (Bonf)
  quote_count: Affan median=0.0 vs 0.0 | ratio=0.0x | r=-0.120 | p_raw=8.835e-20 | p_bonf=9.276e-18 | sig (Bonf)
  view_count: Affan median=3750.0 vs 1360.0 | ratio=2.8x | r=-0.314 | p_raw=1.187e-33 | p_bonf=1.246e-31 | sig (Bonf)
  favorite_count: Affan median=10.0

In [27]:
# Cell 8 — Export ke CSV
# Tiga file output (sekarang sudah include Bonferroni di pairwise):
# 1. descriptive_engagement_per_narrative.csv
# 2. kruskal_wallis_engagement.csv
# 3. pairwise_mannwhitney_engagement.csv  <- sekarang dengan p_raw, p_bonferroni, alpha_bonferroni

# 1. Descriptive
desc_df.to_csv('descriptive_engagement_per_narrative.csv', index=False)
print('Saved: descriptive_engagement_per_narrative.csv')

# 2. Kruskal-Wallis
kw_rows = [{'metric': k, 'H_statistic': round(v['H'], 3), 'p_value': round(v['p'], 6), 'significant': v['sig']}
           for k, v in kruskal_results.items()]
pd.DataFrame(kw_rows).to_csv('kruskal_wallis_engagement.csv', index=False)
print('Saved: kruskal_wallis_engagement.csv')

# 3. Pairwise Mann-Whitney (dengan Bonferroni)
pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv('pairwise_mannwhitney_engagement.csv', index=False)
print('Saved: pairwise_mannwhitney_engagement.csv')
print(f'  -> includes columns: p_value_raw, p_value_bonferroni, alpha_bonferroni, significant_raw, significant_bonferroni')
print(f'  -> alpha_bonferroni = {ALPHA_BONF:.6f} (global correction across {M_TESTS} tests)')

print('\n=== SELESAI ===')

Saved: descriptive_engagement_per_narrative.csv
Saved: kruskal_wallis_engagement.csv
Saved: pairwise_mannwhitney_engagement.csv
  -> includes columns: p_value_raw, p_value_bonferroni, alpha_bonferroni, significant_raw, significant_bonferroni
  -> alpha_bonferroni = 0.000476 (global correction across 105 tests)

=== SELESAI ===
